# COSGD Ablation (M2) — Colab Launcher

Runs the COSGD ablation axes (20.01–20.08) + the scalability and synthetic
dim-sweep scripts on a Colab GPU, writing results to Google Drive so they survive
disconnects. **Idempotent**: re-running resumes (the per-cell `JobManager` skips
completed cells).

Runtime: **GPU**. Run cells top to bottom; re-open and re-run after any drop.

## 1. Clone / pull the repo

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/rayden96/MastersDissertationExperiments.git"
BRANCH   = "m0-infrastructure"   # your working branch
REPO_DIR = "/content/MastersDissertationExperiments"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
print(subprocess.run(["git","-C",REPO_DIR,"log","--oneline","-1"],capture_output=True,text=True).stdout)

## 2. Dependencies

In [ ]:
import importlib, subprocess, sys
for pkg, pip_name in [("torch", None), ("torchvision", None), ("sklearn", "scikit-learn")]:
    try: importlib.import_module(pkg); print(f"{pkg}: ok")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], check=True)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))

## 3. Mount Drive + point results there
The experiment code writes to `get_results_root()`, which honours
`DISSERTATION_RESULTS_ROOT`. Setting it to a Drive folder is all that's needed —
every result (axis runs *and* the master tables) persists to Drive and survives a
session end. No symlinks required.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
RESULTS_ROOT = "/content/drive/MyDrive/dissertation/results"
os.makedirs(RESULTS_ROOT, exist_ok=True)
os.environ["DISSERTATION_RESULTS_ROOT"] = RESULTS_ROOT
print("results ->", RESULTS_ROOT)
# All COSGD results land under {RESULTS_ROOT}/20_cosgd_ablation/... automatically.

## 4. Sanity check

In [ ]:
import sys
sys.path.insert(0, REPO_DIR); sys.path.insert(0, os.path.join(REPO_DIR, "PaperReadyExperiments"))
from common.optimizers import COSGD, GradDrop
print("COSGD wrapper + GradDrop import OK — ready")

## 5. Smoke test (~2 min)

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/20_cosgd_ablation
!python run_all.py --smoke --axes 01 05

## 6. The full M2 sweep (REALIGNED to the scrutiny)
Primary testbed = the **low-dimensional ladder** (iris/wine/breast_cancer/digits)
where COSGD's per-class orthogonalisation has its large, interpretable effect.
Canonical COSGD = the **reclaimed** config (paper full-GS + desc-sort +
`combine=sum` + `combine_norm_cap=2.0`); `mean`/`freq` and the gate are tested
only to show they flatten it. Re-run to resume.

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/20_cosgd_ablation
# mechanism axes on the LOW-DIM LADDER (default datasets) + SGD
!python run_all.py --axes 01 02 03 05 --seeds 2026 2027 2028

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/20_cosgd_ablation
# image check (does the low-dim win carry to conv nets?) + base-optimizer cross
!python run_all.py --axes 01 05 --datasets mnist cifar10 --epochs 10 --seeds 2026 2027 2028
!python run_all.py --axes 06 --seeds 2026 2027 2028
# step-method/BN axis is image-only (BN is the point): defaults to cifar10+cifar100
!python 04_step_method/run.py --epochs 10 --seeds 2026

## 7. Scalability (timing) + synthetic dim-sweep — the COSGD→BoGrad motivation

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/20_cosgd_ablation
!python 07_scalability/run.py                 # synthetic {2..100} subclasses
!python 07_scalability/run.py --real          # CIFAR-10 / EMNIST-47 / CIFAR-100
!python 03_prenormalize/synthetic_dim_sweep.py --trials 50 --max-dim 5000

## 8. Master table + COSGD↔BoGrad mechanism contrast (20.08)

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/20_cosgd_ablation
!python 08_cross_summary/run.py